<a href="https://colab.research.google.com/github/vera2005/MLCV_course1/blob/main/Lesson1/MLCourse_HW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задача 1. Фильр сепии

Цель:
Научиться обрабатывать изображения попиксельно и использовать готовые библиотечные функции для цветовых преобразований.

Дано: изображение (можете загрузить свое, как предусмотрено шаблоном, или использовать готовое аналогично занятию)



Требуется:

1) Реализовать фильтр сепии вручную – для каждого пикселя вычислить новые значения каналов R, G, B по следующим формулам (коэффициенты классической сепии):
R_new = min(255, int(0.393 * R + 0.769 * G + 0.189 * B))

G_new = min(255, int(0.349 * R + 0.686 * G + 0.168 * B))

B_new = min(255, int(0.272 * R + 0.534 * G + 0.131 * B))

где R, G, B – исходные значения (0–255) каждого пикселя.

2) Реализовать тот же фильтр с использованием библиотечной функции – применить матричное преобразование с помощью cv2.transform().
Матрица для сепии

sepia_matrix = np.array(

[[0.393, 0.769, 0.189],

[0.349, 0.686, 0.168],

[0.272, 0.534, 0.131]])

Подсказка: После применения матрицы необходимо обрезать значения до диапазона [0, 255] и привести к типу uint8.

In [ ]:
import cv2
import numpy as np
from google.colab import files
import matplotlib.pyplot as plt

In [ ]:
# ---------- 1. Загрузка изображения ----------
uploaded = files.upload()
filename = next(iter(uploaded))
img_bgr = cv2.imread(filename)          # OpenCV загружает в BGR
plt.figure(figsize=(15, 5))
plt.imshow(img_bgr)
plt.axis('off')
plt.show()

Переведите изображение в цветовое пространство RGB

In [ ]:
img_rgb =

In [ ]:
plt.figure(figsize=(15, 5))
plt.imshow(img_rgb)
plt.title('Оригинал')
plt.axis('off')
plt.show()

Место для ручной, рабоче-крестьянской реализации фильтра (попиксельно). Реализуйте преобразование в функции def sepia_manual(img), в качестве результата верните новое изображение.

Подсказка по идее: считать размеры изображения. Вложенным циклом проходить по каждому пикселю и для каждого цветого канала (R, G, B) высчитывать новое значения по формулам выше. В качестве результата работы функции вернуть новое изображение

In [ ]:
def sepia_manual(img):


img_manual = sepia_manual(img_rgb)

Библиотечная реализация, через матрицу в OpenCV

In [ ]:
sepia_matrix = np.array([[0.393, 0.769, 0.189],
                         [0.349, 0.686, 0.168],
                         [0.272, 0.534, 0.131]], dtype=np.float32)

img_lib =

In [ ]:
# Визуализация
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(img_rgb)
plt.title('Оригинал')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(img_manual)
plt.title('Ручная сепия')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(img_lib)
plt.title('Библиотечная сепия (cv2.transform)')
plt.axis('off')

plt.show()

# Задача 2. Построение нейрона-детектора красных оттенков.
Цель: Научиться подбирать веса и смещение для одного искусственного нейрона (перцептрона) с пороговой функцией активации, чтобы он корректно классифицировал заданный набор цветов как «красный» (1) или «не красный» (0).

Дано:

20 цветов, представленных в формате RGB (каждый компонент от 0 до 255).

Для каждого цвета указано, должен ли он попадать под детекцию красного (метка 1) или нет (метка 0).

Список цветов с их метками приведён в коде ниже.

Задача: написать класс нейрона и подобрать такие значения весов, чтобы нейрон правильно классифицировал оттенки красного.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# -------------------- ДАННЫЕ --------------------
# 20 цветов (R, G, B) и их метки (1 - красный, 0 - нет)
colors = [
    # Красные оттенки (метка 1)
    (255, 0, 0),     # 1 - чистый красный
    (200, 0, 0),     # 1 - тёмно-красный
    (255, 50, 50),   # 1 - светло-красный
    (255, 100, 0),   # 1 - оранжевый (красный с жёлтым)
    (255, 150, 0),   # 1 - светло-оранжевый
    (180, 50, 50),   # 1 - бордовый
    (128, 0, 0),     # 1 - тёмно-бордовый
    (150, 0, 50),    # 1 - вишнёвый (красный с синим оттенком)
    (200, 80, 80),   # 1 - розовато-красный (всё ещё красный)
    # Не красные (метка 0)
    (255, 200, 200), # 0 - розовый
    (0, 255, 0),     # 0 - зелёный
    (0, 0, 255),     # 0 - синий
    (255, 255, 0),   # 0 - жёлтый
    (0, 255, 255),   # 0 - голубой
    (255, 0, 255),   # 0 - пурпурный (магента)
    (100, 100, 100), # 0 - серый
    (0, 0, 0),       # 0 - чёрный
    (255, 255, 255), # 0 - белый
    (100, 200, 100), # 0 - салатовый
    (50, 50, 150),   # 0 - тёмно-синий
]

# Метки: 1 для первых 10, 0 для последних 10
labels = [1]*9 + [0]*11

В ячейке ниже напишите класс нейрона, создайте экземпляр my_neuron с заданными значениями весов и смещения.

Подсказка 1: т.к. мы работает с тремя цветовыми каналами, то весов будет 3 (+ смещение)

Подсказка 2: в качестве функции активации попробуйте применить пороговую

In [ ]:
my_neuron =

Проверьте корректность работы
вашего нейрона при помощи функции ниже (просто запустите код ячейки)

In [ ]:
# Функция проверки
def test_neuron(neuron):
    correct = 0
    errors = []
    for i, (R, G, B) in enumerate(colors):
        pred = neuron.predict(R, G, B)
        if pred == labels[i]:
            correct += 1
        else:
            errors.append((i, R, G, B, labels[i], pred))

    accuracy = correct / len(colors) * 100
    print(f"Точность: {accuracy:.1f}% ({correct}/{len(colors)})")

    if errors:
        print("Ошибки на цветах (индекс, (R,G,B), ожидалось, получено):")
        for e in errors:
            print(e)
    else:
        print("✅ ВЕРНО! Все цвета классифицированы правильно.")


# Проверяем
test_neuron(my_neuron)